<a href="https://colab.research.google.com/github/mydas-denzel/AIRecon-Stuff/blob/main/scripts/airecon_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIRecon + Ollama on Google Colab

Deploy **Ollama** on a Colab GPU and connect your local **AIRecon** to it via a public tunnel.

| Step | What happens |
|------|--------------|
| 1 | Check GPU + VRAM available |
| 2 | Install Ollama (CUDA-accelerated) |
| 3 | Start Ollama server |
| 4 | Pull best model for your GPU |
| 5 | Open Cloudflare Quick Tunnel (no account needed) |
| 6 | Generate `~/.airecon/config.yaml` snippet |
| 7 | Keep-alive loop (prevents Colab idle disconnect) |

**Requirements**
- Colab GPU runtime: `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro)
- Local machine with AIRecon installed
- Run cells **top to bottom**, only once per session

> If the Colab tab disconnects, re-run from **Cell 3** (Ollama is already installed).

## Cell 1 — GPU & Environment Check

In [1]:
import datetime
import os
import re
import shutil
import subprocess
import sys
import threading
import time

import requests as _req

# ── GPU info ──────────────────────────────────────────────────
def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

smi = _run(['nvidia-smi',
            '--query-gpu=name,memory.total,memory.free,driver_version',
            '--format=csv,noheader'])

if smi.returncode != 0:
    print('ERROR: No GPU detected. Change runtime: Runtime → Change runtime type → GPU')
    sys.exit(1)

gpu_fields = [f.strip() for f in smi.stdout.strip().split(',')]
GPU_NAME   = gpu_fields[0]
VRAM_TOTAL = int(gpu_fields[1].replace(' MiB', '')) / 1024   # GB
VRAM_FREE  = int(gpu_fields[2].replace(' MiB', '')) / 1024   # GB
DRIVER_VER = gpu_fields[3]

print(f'GPU         : {GPU_NAME}')
print(f'VRAM total  : {VRAM_TOTAL:.1f} GB')
print(f'VRAM free   : {VRAM_FREE:.1f} GB')
print(f'Driver      : {DRIVER_VER}')

# ── CUDA version ─────────────────────────────────────────────
nvcc = _run(['nvcc', '--version'])
cuda_ver = 'unknown'
if nvcc.returncode == 0:
    m = re.search(r'release ([\d.]+)', nvcc.stdout)
    if m:
        cuda_ver = m.group(1)
print(f'CUDA        : {cuda_ver}')

# ── Disk space ───────────────────────────────────────────────
df = _run(['df', '-h', '/'])
disk_line = df.stdout.splitlines()[1] if df.returncode == 0 else ''
disk_parts = disk_line.split()
if len(disk_parts) >= 4:
    print(f'Disk free   : {disk_parts[3]} / {disk_parts[1]}')

print()
print('Environment OK — continue to Cell 2.')


GPU         : Tesla T4
VRAM total  : 15.0 GB
VRAM free   : 14.6 GB
Driver      : 580.82.07
CUDA        : 12.8
Disk free   : 66G / 113G

Environment OK — continue to Cell 2.


## Cell 2 — Install Ollama

In [2]:
def _run_stream(cmd, shell=False):
    proc = subprocess.Popen(
        cmd, shell=shell,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

# ── Install zstd (required by Ollama installer) ───────────────
print('Installing zstd (required by Ollama)...')
rc = _run_stream(['apt-get', 'install', '-y', '-q', 'zstd'])
if rc != 0:
    print('WARNING: apt-get zstd failed, trying anyway...')

# ── Install Ollama ────────────────────────────────────────────
if shutil.which('ollama'):
    ver = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
    print(f'Ollama already installed: {ver.stdout.strip()}')
else:
    print('Downloading and installing Ollama (CUDA-accelerated)...')
    rc = _run_stream('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
    if rc == 0:
        ver = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
        print(f'\nOllama installed: {ver.stdout.strip()}')
    else:
        print('\nERROR: Ollama install failed.')
        raise RuntimeError('Ollama install failed')

print('Done — continue to Cell 3.')


Installing zstd (required by Ollama)...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,089 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ..

## Cell 3 — Start Ollama Server

In [3]:
OLLAMA_PORT = 11434
OLLAMA_API  = f'http://127.0.0.1:{OLLAMA_PORT}'

def _ollama_alive():
    try:
        r = _req.get(f'{OLLAMA_API}/api/version', timeout=3)
        return r.ok, r.json().get('version', '?')
    except Exception:
        return False, ''

alive, ver = _ollama_alive()
if alive:
    print(f'Ollama already running (v{ver})')
else:
    env = os.environ.copy()
    env.update({
        'OLLAMA_HOST'           : f'0.0.0.0:{OLLAMA_PORT}',
        'OLLAMA_ORIGINS'        : '*',
        'OLLAMA_KEEP_ALIVE'     : '-1',
        'OLLAMA_NUM_PARALLEL'   : '1',
        'OLLAMA_FLASH_ATTENTION': '1',
        'CUDA_VISIBLE_DEVICES'  : '0',
    })
    OLLAMA_PROC = subprocess.Popen(
        ['ollama', 'serve'],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print('Starting Ollama', end='', flush=True)
    for _ in range(40):
        time.sleep(1)
        alive, ver = _ollama_alive()
        if alive:
            break
        print('.', end='', flush=True)
    else:
        print('\nERROR: Ollama did not start.')
        raise RuntimeError('Ollama did not start')

    print(f'\nOllama running (v{ver})')

print('Done — continue to Cell 4.')


Starting Ollama
Ollama running (v0.24.0)
Done — continue to Cell 4.


## Cell 4 — Model Selection & Pull

The cell auto-detects VRAM and picks the best fitting model.  
Override `MODEL_TO_PULL` if you want a specific model.

In [ ]:
MODEL_CATALOGUE = [
    {'name': 'qwen3.5:9b',       'vram_gb':  6.0,  'ctx_size': 32768,  'desc': 'Qwen3.5 9B    — minimum viable, expect frequent errors'},
    {'name': 'qwen3.5:35b-a3b',  'vram_gb': 16.0,  'ctx_size': 65536,  'desc': 'Qwen3.5 35B MoE — lower VRAM than dense 35B'},
    {'name': 'qwen3.5:35b',      'vram_gb': 20.0,  'ctx_size': 65536,  'desc': 'Qwen3.5 35B   — recommended for most users'},
    {'name': 'qwen3.5:122b',     'vram_gb': 48.0,  'ctx_size': 65536,  'desc': 'Qwen3.5 122B  — best quality, most reliable'},
]

# ── Pick best fitting model ───────────────────────────────────
fitting = [m for m in MODEL_CATALOGUE if VRAM_FREE >= m['vram_gb']]

print('Models available for your GPU:')
for m in MODEL_CATALOGUE:
    fits = 'OK' if VRAM_FREE >= m['vram_gb'] else '--'
    print(f'  [{fits}] {m["name"]:22s}  {m["vram_gb"]:4.0f} GB  {m["desc"]}')

SELECTED = fitting[-1] if fitting else MODEL_CATALOGUE[0]
print(f'\nAuto-selected: {SELECTED["name"]}  (ctx_size={SELECTED["ctx_size"]})')
print()

# ── OVERRIDE HERE ─────────────────────────────────────────────
# MODEL_TO_PULL = SELECTED['name']
# MODEL_TO_PULL = 'qwen3.5:9b'
# MODEL_TO_PULL = 'qwen3.5:35b-a3b'
MODEL_TO_PULL = 'qwen3.5:35b'
# MODEL_TO_PULL = 'qwen3.5:122b'

chosen_meta = next((m for m in MODEL_CATALOGUE if m['name'] == MODEL_TO_PULL), SELECTED)
CTX_SIZE = chosen_meta['ctx_size']

# ── Check if already pulled ───────────────────────────────────
tags = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
already_have = MODEL_TO_PULL in tags.stdout

if already_have:
    print(f'Model {MODEL_TO_PULL} already pulled — skipping download.')
else:
    print(f'Pulling {MODEL_TO_PULL} — this may take several minutes...')
    proc = subprocess.Popen(
        ['ollama', 'pull', MODEL_TO_PULL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    last = ''
    for line in proc.stdout:
        line = line.rstrip()
        if line and line != last:
            if any(c in line for c in ['▓', '░', '%']):
                print(f'\r  {line[:90]}', end='', flush=True)
            else:
                print(f'\n  {line}', flush=True)
            last = line
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Model pull failed for {MODEL_TO_PULL}')
    print('\nPull complete.')

# ── Smoke test via REST API ───────────────────────────────────
print(f'\nLoading {MODEL_TO_PULL} into VRAM (first load may take a few minutes)...', flush=True)
_load_start = time.time()
try:
    resp = _req.post(
        f'{OLLAMA_API}/api/generate',
        json={'model': MODEL_TO_PULL, 'prompt': 'Reply with: OK', 'stream': False},
        timeout=600
    )
    _elapsed = int(time.time() - _load_start)
    if resp.ok:
        reply = resp.json().get('response', '').strip()
        print(f'  Model response ({_elapsed}s): {reply[:80]}')
        print('Model OK.')
    else:
        print(f'  HTTP {resp.status_code}: {resp.text[:200]}')
        print('WARNING: Unexpected response — model may still be usable.')
except _req.exceptions.Timeout:
    _elapsed = int(time.time() - _load_start)
    print(f'  Timed out after {_elapsed}s.')
    print('WARNING: Model did not respond in 10 min. Try a smaller model.')
except Exception as e:
    print(f'  Error: {e}')
    print('WARNING: Could not verify model — continuing anyway.')

print('\nDone — continue to Cell 5.')


Models available for your GPU:
  [OK] qwen3.5:9b                 6 GB  Qwen3.5 9B    — minimum viable, expect frequent errors
  [--] qwen3.5:35b-a3b           16 GB  Qwen3.5 35B MoE — lower VRAM than dense 35B
  [--] qwen3.5:35b               20 GB  Qwen3.5 35B   — recommended for most users
  [--] qwen3.5:122b              48 GB  Qwen3.5 122B  — best quality, most reliable

Auto-selected: qwen3.5:9b  (ctx_size=32768)

Pulling qwen3.5:9b — this may take several minutes...

  pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
  pulling dec52a44569a:  34% ▕██████            ▏ 2.3 GB/6.6 GB  188 MB/s     22s[

## Cell 5 — Cloudflare Quick Tunnel (Recommended — no account needed)

Creates a public `*.trycloudflare.com` URL that proxies to `localhost:11434`.  
No Cloudflare account or login required.

**Alternative:** If you prefer ngrok, skip to Cell 5B.

In [ ]:
CF_BIN = '/usr/local/bin/cloudflared'

# ── Download cloudflared binary ───────────────────────────────
if not os.path.exists(CF_BIN):
    print('Downloading cloudflared...')
    dl = subprocess.run([
        'wget', '-q', '-O', CF_BIN,
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], capture_output=True, text=True)
    if dl.returncode != 0:
        print('ERROR: wget failed:', dl.stderr)
        raise RuntimeError('cloudflared download failed')
    os.chmod(CF_BIN, 0o755)
    print('cloudflared downloaded.')
else:
    print('cloudflared already present.')

# ── Verify Ollama is reachable locally before opening tunnel ──
try:
    r = _req.get(f'{OLLAMA_API}/api/version', timeout=5)
    _ver = r.json().get('version', '?')
    print(f'Ollama local OK: v{_ver}')
except Exception as e:
    print(f'ERROR: Ollama not reachable locally ({e})')
    print('Re-run Cell 3 first.')
    raise RuntimeError('Ollama not running') from e

# ── Launch tunnel ─────────────────────────────────────────────
TUNNEL_URL = None
_cf_log    = []

def _run_cloudflared():
    global TUNNEL_URL
    proc = subprocess.Popen(
        [CF_BIN, 'tunnel',
         '--url', f'http://127.0.0.1:{OLLAMA_PORT}',
         '--no-autoupdate',
         '--loglevel', 'info'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        line = line.strip()
        _cf_log.append(line)
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m and TUNNEL_URL is None:
            TUNNEL_URL = m.group(0)

threading.Thread(target=_run_cloudflared, daemon=True).start()

print('Waiting for Cloudflare tunnel URL', end='', flush=True)
for _ in range(60):
    if TUNNEL_URL:
        break
    time.sleep(1)
    print('.', end='', flush=True)
else:
    print('\nERROR: Tunnel URL not detected. Last log:')
    print('\n'.join(_cf_log[-15:]))
    raise RuntimeError('Cloudflare tunnel failed — try Cell 5B (ngrok) instead.')

print(f'\n\nTunnel active : {TUNNEL_URL}')
print(f'Ollama local  : {OLLAMA_API}  [verified OK]')
print()

# ── Smoke test: confirm streaming works on localhost ─────────
print('Testing chunked streaming on localhost...', end='', flush=True)
try:
    with _req.post(
        f'{OLLAMA_API}/api/generate',
        json={'model': MODEL_TO_PULL, 'prompt': 'Say: OK', 'stream': True},
        stream=True, timeout=60
    ) as sr:
        chunks = 0
        for chunk in sr.iter_lines():
            if chunk:
                chunks += 1
                if chunks >= 3:
                    break
    print(f' OK ({chunks} chunks received)')
except Exception as e:
    print(f' WARNING: {e}')

print()
print('Done — continue to Cell 6.')
print()
print('Verify from your local machine:')
print(f'  curl {TUNNEL_URL}/api/version')
print(f'  curl {TUNNEL_URL}/api/tags')


## Cell 5B — ngrok Tunnel (Alternative, skipped by default)

**This cell is skipped automatically when using "Run All"** (`USE_NGROK = False`).

Use ngrok only if Cell 5 (cloudflared) failed. To activate:
1. Set `USE_NGROK = True` in the cell below
2. Set `NGROK_AUTH_TOKEN = 'your-token'`
3. Get a free token at https://dashboard.ngrok.com/signup


In [ ]:
# ── Cell 5B is SKIPPED by default when using "Run All" ───────
# Set USE_NGROK = True only if Cell 5 (cloudflared) failed.
USE_NGROK = False
# ─────────────────────────────────────────────────────────────

if not USE_NGROK:
    print('Cell 5B skipped (USE_NGROK=False). TUNNEL_URL from Cell 5 is used.')
else:
    # ── FILL IN YOUR NGROK AUTH TOKEN ────────────────────────
    NGROK_AUTH_TOKEN = ''   # <-- paste your token here
    # ─────────────────────────────────────────────────────────

    if not NGROK_AUTH_TOKEN:
        print('ERROR: Set NGROK_AUTH_TOKEN above.')
        print('Get a free token at https://dashboard.ngrok.com/signup')
        raise ValueError('NGROK_AUTH_TOKEN not set')

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)

    from pyngrok import ngrok, conf  # noqa: PLC0415

    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.kill()
    time.sleep(1)

    tunnel_ng  = ngrok.connect(OLLAMA_PORT, 'http', bind_tls=True)
    TUNNEL_URL = tunnel_ng.public_url.replace('http://', 'https://')

    print(f'ngrok tunnel active: {TUNNEL_URL}')
    print('Done — continue to Cell 6.')


## Cell 6 — Generate AIRecon Configuration

In [ ]:
# ── Compute recommended settings ─────────────────────────────
# Increase timeouts for network latency (local Ollama is fast; tunnel adds ~200ms)
OLLAMA_TIMEOUT       = 300
OLLAMA_CHUNK_TIMEOUT = 300

# Reduce num_predict slightly to avoid very long waits over tunnel
OLLAMA_NUM_PREDICT   = 8192

CTX_SMALL = CTX_SIZE // 2

# ── Print YAML config snippet ─────────────────────────────────
CONFIG_YAML = f"""# ── Paste this into ~/.airecon/config.yaml ──────────────────
# Generated by airecon_colab.ipynb
# GPU: {GPU_NAME}  VRAM: {VRAM_TOTAL:.1f}GB

ollama_url: "{TUNNEL_URL}"
ollama_model: "{MODEL_TO_PULL}"
ollama_num_ctx: {CTX_SIZE}
ollama_num_ctx_small: {CTX_SMALL}
ollama_timeout: {OLLAMA_TIMEOUT}.0
ollama_chunk_timeout: {OLLAMA_CHUNK_TIMEOUT}.0
ollama_temperature: 0.15
ollama_num_predict: {OLLAMA_NUM_PREDICT}
# ──────────────────────────────────────────────────────────────
"""

ENV_VARS = f"""# Or export these in your terminal before running airecon:
export AIRECON_OLLAMA_URL="{TUNNEL_URL}"
export AIRECON_OLLAMA_MODEL="{MODEL_TO_PULL}"
export AIRECON_OLLAMA_TIMEOUT="{OLLAMA_TIMEOUT}"
export AIRECON_OLLAMA_CHUNK_TIMEOUT="{OLLAMA_CHUNK_TIMEOUT}"
"""

separator = '=' * 65
print(separator)
print('COPY THIS CONFIG TO YOUR LOCAL MACHINE')
print(separator)
print(CONFIG_YAML)
print(separator)
print(ENV_VARS)
print(separator)
print()
print(f'Tunnel URL  : {TUNNEL_URL}')
print(f'Model       : {MODEL_TO_PULL}')
print(f'Context     : {CTX_SIZE:,} tokens')
print(f'GPU         : {GPU_NAME}  ({VRAM_TOTAL:.1f} GB)')
print()

# ── Test from local: quick one-liner ─────────────────────────
print('Test from your local machine:')
print(f'  curl {TUNNEL_URL}/api/version')
print(f'  curl -s {TUNNEL_URL}/api/tags | python3 -m json.tool')
print()
print('Done — continue to Cell 7 (keep-alive).')

## Cell 7 — Keep-Alive Monitor

Colab disconnects after **~90 minutes of inactivity** (kernel idle).  
This cell pings Ollama and the tunnel every 60 seconds to prevent that.

**Leave this cell running.** The tunnel URL is printed every 5 minutes.

In [ ]:
PING_INTERVAL_SEC = 55
BANNER_INTERVAL   = 5 * 60

_stats = {'ok': 0, 'err': 0, 'start': time.time()}
_last_banner = 0

def _ping():
    global _last_banner
    while True:
        now     = time.time()
        elapsed = int(now - _stats['start'])
        h, rem  = divmod(elapsed, 3600)
        m, s    = divmod(rem, 60)
        ts      = datetime.datetime.now().strftime('%H:%M:%S')
        uptime  = f'{h:02d}:{m:02d}:{s:02d}'

        try:
            r = _req.get(f'{OLLAMA_API}/api/version', timeout=5)
            ollama_ok = r.ok
        except Exception:
            ollama_ok = False

        if ollama_ok:
            _stats['ok'] += 1
            status = 'OK'
        else:
            _stats['err'] += 1
            status = f'WARN  ollama={ollama_ok}'

        print(f'[{ts}] [{status}] uptime={uptime}  pings=ok:{_stats["ok"]} err:{_stats["err"]}')

        if (now - _last_banner) >= BANNER_INTERVAL:
            _last_banner = now
            print()
            print('  Tunnel URL :', TUNNEL_URL)
            print('  Model      :', MODEL_TO_PULL)
            print()

        time.sleep(PING_INTERVAL_SEC)

_keep_alive_thread = threading.Thread(target=_ping, daemon=True)
_keep_alive_thread.start()

print('Keep-alive monitor started.')
print(f'Tunnel URL : {TUNNEL_URL}')
print(f'Model      : {MODEL_TO_PULL}')
print()
print('This cell must keep running. Use Colab menu: Runtime → Interrupt if you need to stop.')
print()

_keep_alive_thread.join()


## Reconnect after Disconnect

If the Colab session disconnects:

1. Click **Connect** (top right)
2. Re-run **Cell 3** (start Ollama — install is persistent within the session)
3. Re-run **Cell 5** (new tunnel URL will be generated)
4. Re-run **Cell 6** to get the new config snippet
5. Re-run **Cell 7**
6. Update `ollama_url` in `~/.airecon/config.yaml` with the new URL

> The model stays in memory after reconnect as long as Ollama is still running.

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| `RuntimeError: Ollama did not start` | Re-run Cell 3; check GPU runtime is enabled |
| Tunnel URL not found | Try ngrok (Cell 5B) instead of cloudflared |
| `curl` returns 502/503 | Tunnel is warming up — wait 10s and retry |
| Model pull fails | Check disk space (`df -h /`) — need 6–50 GB free depending on model |
| AIRecon timeout errors | Increase `ollama_timeout` and `ollama_chunk_timeout` to 600 |
| Slow responses | Normal over tunnel — T4 does ~15–30 tok/s for 9B; slower for 35B |
| `Invalid IPv6 URL` on AIRecon | Update AIRecon (bug fixed in current version) |
| qwen3.5:9b gives bad output | Expected — use 35B or 35B-MoE for real hunts |

### GPU → Model recommendation

| Colab GPU | VRAM | Recommended model |
|-----------|------|-------------------|
| T4 (free) | 15 GB | `qwen3.5:9b` only option |
| L4 (Pro) | 22 GB | `qwen3.5:35b-a3b` (MoE) |
| A100 (Pro+) | 40 GB | `qwen3.5:35b` |
| H100 (Pro+) | 80 GB | `qwen3.5:122b` |

### Verify tunnel from local machine

```bash
# Basic health check
curl https://YOUR-TUNNEL.trycloudflare.com/api/version

# List pulled models
curl -s https://YOUR-TUNNEL.trycloudflare.com/api/tags | python3 -m json.tool

# Test inference (non-streaming)
curl https://YOUR-TUNNEL.trycloudflare.com/api/generate \
     -d '{"model":"qwen3.5:9b","prompt":"hello","stream":false}'
```

### AIRecon config location

```
~/.airecon/config.yaml
```

Set `ollama_url` to the tunnel URL (no trailing slash).
